# Жизненный цикл и карточка моделей — отбор напрямую из Библиотеки моделей

> **Версия отбора: все версии моделей, находящиеся в эксплуатации.**


Этот ноутбук формирует периметр моделей непосредственно из Библиотеки моделей, без внешних файлов со списком версий или статусами внедрения.

Критически важно:

- исходный перечень `model_ver_sid` формируется непосредственно из Библиотеки моделей;
- в периметр попадают только версии моделей, для которых актуальный признак `model_ver_prom_expl_flag = true` (версия находится в эксплуатации);
- категория значимости не влияет на отбор: сохраняются все версии в эксплуатации, включая E, пустые и прочие значения;
- связанные `busn_task_sid` и `implm_sid` формируются внутри ноутбука из Hive только для окончательно выбранных версий моделей;
- последние SCD-записи и связанные сущности выбираются по `start_dt DESC`, как в исходных SQL-блоках;
- при одинаковом максимальном `start_dt` сохраняется исходное поведение `ROW_NUMBER()` без дополнительного критерия сортировки.

Промежуточные Hive-таблицы `DDD_*` не создаются: это сокращает код, но не меняет отбор строк или значений.


In [ ]:
import os
import sys

os.environ["SPARK_MAJOR_VERSION"] = "3.5.1"
os.environ["SPARK_HOME"] = "/usr/sdp/current/spark3.5.1-client/"
os.environ["PYSPARK_DRIVER"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
sys.path.insert(0, "/usr/sdp/current/spark3.5.1-client/python/")
sys.path.insert(0, "/usr/sdp/current/spark3.5.1-client/python/lib/py4j-0.10.9.7-src.zip")

from functools import reduce
from pathlib import Path

import pandas as pd
from pyspark import SparkConf
from pyspark.sql import DataFrame, SparkSession, Window, functions as F

conf = (
    SparkConf()
    .setAppName("model_lifecycle_and_card_exact")
    .setMaster("yarn")
    .set("spark.executor.cores", "2")
    .set("spark.executor.memory", "6g")
    .set("spark.executor.memoryOverhead", "1g")
    .set("spark.driver.memory", "6g")
    .set("spark.driver.maxResultSize", "4g")
    .set("spark.shuffle.service.enabled", "true")
    .set("spark.hadoop.mapreduce.input.fileinputformat.input.dir.recursive", "true")
    .set("spark.dynamicAllocation.enabled", "true")
    .set("spark.dynamicAllocation.executorIdleTimeout", "120s")
    .set("spark.dynamicAllocation.cachedExecutorIdleTimeout", "600s")
    .set("spark.dynamicAllocation.initialExecutors", "4")
    .set("spark.dynamicAllocation.maxExecutors", "12")
    .set("spark.dynamicAllocation.shuffleTracking.enabled", "true")
    .set("spark.port.maxRetries", "150")
    .set("spark.sql.parquet.int96RebaseModeInWrite", "CORRECTED")
    .set("spark.sql.parquet.writeLegacyFormat", "true")
    .set("spark.sql.parquet.compression.codec", "snappy")
)

spark = SparkSession.builder.config(conf=conf).enableHiveSupport().getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)


In [ ]:
# Единственное место настройки путей и источника.
SOURCE_DB = "prx_pri_custom_ris_l_library_custom_risk_model_library"
OUTPUT_DIR = Path.cwd().resolve()

HISTORY_CUTOFF = "2025-10-02 00:00:00"
MAX_EXCEL_ROWS = 1_048_575

TABLES = {
    "model": f"{SOURCE_DB}.t_model",
    "model_ver": f"{SOURCE_DB}.t_model_ver",
    "model_ver_anlt_dtl": f"{SOURCE_DB}.t_model_ver_anlt_dtl",
    "model_ver_prom": f"{SOURCE_DB}.t_model_ver_prom",
    "prom_link": f"{SOURCE_DB}.t_model_ver_prom_x_prom_implm",
    "busn_link": f"{SOURCE_DB}.t_model_ver_x_busn_task",
    "busn_task": f"{SOURCE_DB}.t_busn_task",
    "valid": f"{SOURCE_DB}.t_valid",
    "valid_it": f"{SOURCE_DB}.t_valid_it",
    "manual_link": f"{SOURCE_DB}.t_model_ver_x_montrg_manual",
    "manual_result": f"{SOURCE_DB}.t_montrg_manual_rslt",
    "auto_link": f"{SOURCE_DB}.t_model_ver_x_montrg_auto",
    "auto_monitor": f"{SOURCE_DB}.t_montrg_auto",
    "change_log": f"{SOURCE_DB}.t_ent_param_chg",
}


## Входные ID для связанных сущностей


In [ ]:
# Внешние файлы со списками ID не используются.
# model_ver_sid, busn_task_sid и implm_sid формируются ниже напрямую из Hive.
print("Внешние списки ID не требуются: периметр и связанные сущности будут построены из Hive")


## Источники и выбор последней строки строго по `start_dt`


In [ ]:
def require_table(table_name: str) -> None:
    if not spark.catalog.tableExists(table_name):
        raise RuntimeError(f"Не найдена таблица: {table_name}")


def require_columns(df: DataFrame, table_name: str, columns) -> None:
    missing = sorted(set(columns) - set(df.columns))
    if missing:
        raise RuntimeError(f"{table_name}: отсутствуют поля: {', '.join(missing)}")


def take_latest(df: DataFrame, keys, order_column: str, label: str) -> DataFrame:
    """ROW_NUMBER() OVER (PARTITION BY keys ORDER BY order_column DESC), как в исходной логике."""
    window = Window.partitionBy(*keys).orderBy(F.col(order_column).desc())
    return (
        df.withColumn("__rn", F.row_number().over(window))
        .filter(F.col("__rn") == 1)
        .drop("__rn")
    )


for table_name in TABLES.values():
    require_table(table_name)

model_raw = spark.table(TABLES["model"])
model_ver_raw = spark.table(TABLES["model_ver"])
anlt_raw = spark.table(TABLES["model_ver_anlt_dtl"])
prom_raw = spark.table(TABLES["model_ver_prom"])
prom_link_raw = spark.table(TABLES["prom_link"])
busn_link_raw = spark.table(TABLES["busn_link"])
busn_raw = spark.table(TABLES["busn_task"])
valid_raw = spark.table(TABLES["valid"])
valid_it_raw = spark.table(TABLES["valid_it"])
manual_link_raw = spark.table(TABLES["manual_link"])
manual_result_raw = spark.table(TABLES["manual_result"])
auto_link_raw = spark.table(TABLES["auto_link"])
auto_raw = spark.table(TABLES["auto_monitor"])
change_raw = spark.table(TABLES["change_log"])

required = {
    TABLES["model"]: ["model_sid", "start_dt", "model_name", "model_rsk_flag", "model_rsk_type_name", "model_rsk_sgmnt_name", "model_type_name", "model_subtype_name", "model_code", "model_conf_ctgry_name", "model_sel_secret_flag"],
    TABLES["model_ver"]: ["model_sid", "model_ver_sid", "start_dt", "model_ver_signfcnt_lvl_name", "model_ver_dev_start_fact_dttm", "model_ver_signfcnt_ctgry_code", "model_ver_signfcnt_ctgry_descr_txt", "model_ver_dev_end_fact_dttm", "model_ver_dev_sys_name", "model_ver_data_mart_link_txt", "model_ver_crtn_dttm", "model_ver_dev_report_sid", "model_ver_prevalid_report_file_sid", "model_ver_prevalid_report_link_sid", "model_ver_prevalid_report_link_txt", "model_ver_dev_block_name", "model_ver_dev_dprtmt_name", "model_ver_prevalid_dttm"],
    TABLES["model_ver_anlt_dtl"]: ["model_ver_sid", "start_dt", "model_ver_stts_name", "model_stts_name", "model_ver_prom_expl_flag"],
    TABLES["model_ver_prom"]: ["model_ver_sid", "model_ver_prom_sid", "start_dt", "model_ver_prom_instr_name", "model_ver_prom_sys_name", "model_ver_prom_stts_name", "model_ver_prom_crtn_dttm", "model_ver_prom_dev_block_name", "model_ver_prom_dev_dprtmt_name"],
    TABLES["busn_link"]: ["model_ver_sid", "busn_task_sid", "start_dt"],
    TABLES["busn_task"]: ["busn_task_sid", "start_dt", "busn_task_name", "busn_task_claim_descr_txt", "busn_task_crtn_dttm", "busn_task_employer_block_name", "busn_task_employer_dprtmt_name"],
    TABLES["valid"]: ["model_ver_sid", "valid_sid", "start_dt", "valid_crtn_dttm", "valid_report_sid", "valid_dprtmt_name"],
    TABLES["manual_link"]: ["model_ver_sid", "montrg_manual_sid", "start_dt"],
    TABLES["manual_result"]: ["montrg_manual_sid", "start_dt", "montrg_manual_rslt_end_dttm", "montrg_manual_rslt_start_dttm", "montrg_manual_rslt_report_sid"],
    TABLES["auto_link"]: ["model_ver_sid", "montrg_auto_sid", "start_dt"],
    TABLES["auto_monitor"]: ["montrg_auto_sid", "start_dt", "montrg_auto_crtn_dttm", "montrg_auto_end_dttm", "montrg_auto_dprtmt_name"],
    TABLES["valid_it"]: ["model_ver_prom_sid", "valid_it_sid", "start_dt", "valid_it_end_dttm", "valid_it_start_dttm", "valid_it_crtn_dttm", "valid_it_rslt_name", "valid_it_dprtmt_name"],
    TABLES["prom_link"]: ["model_ver_prom_sid", "prom_implm_sid", "start_dt"],
    TABLES["change_log"]: ["ent_sid", "ent_type_name", "ent_param_chg_sid", "ent_param_chg_val", "ent_param_chg_usr_name", "start_dttm", "end_dttm"],
}
for table_name, columns in required.items():
    require_columns(spark.table(table_name), table_name, columns)

print("Все таблицы и обязательные поля найдены")


## Карточка: та же последовательность последних связанных записей


In [ ]:
# В исходнике t_model и t_model_ver сначала сводятся к максимальному start_dt по model_sid.
model_latest = take_latest(model_raw, ["model_sid"], "start_dt", "t_model по model_sid")
model_ver_latest = take_latest(model_ver_raw, ["model_sid"], "start_dt", "t_model_ver по model_sid")

card0 = (
    model_ver_latest.alias("mv")
    .join(model_latest.alias("m"), F.col("mv.model_sid") == F.col("m.model_sid"), "inner")
    .select(
        F.col("m.model_name"), F.col("m.model_rsk_flag"), F.col("m.model_rsk_type_name"),
        F.col("m.model_rsk_sgmnt_name"), F.col("m.model_type_name"), F.col("m.model_subtype_name"),
        F.col("m.model_code"), F.col("m.model_conf_ctgry_name"), F.col("m.model_sel_secret_flag"),
        F.col("m.model_sid"), F.col("mv.model_ver_signfcnt_lvl_name"),
        F.col("mv.model_ver_dev_start_fact_dttm"), F.col("mv.model_ver_signfcnt_ctgry_code"),
        F.col("mv.model_ver_signfcnt_ctgry_descr_txt"), F.col("mv.model_ver_dev_end_fact_dttm"),
        F.col("mv.model_ver_dev_sys_name"), F.col("mv.model_ver_data_mart_link_txt"),
        F.col("mv.model_ver_crtn_dttm"), F.col("mv.model_ver_dev_report_sid"),
        F.col("mv.model_ver_prevalid_report_file_sid"), F.col("mv.model_ver_prevalid_report_link_sid"),
        F.col("mv.model_ver_prevalid_report_link_txt"), F.col("mv.model_ver_dev_block_name"),
        F.col("mv.model_ver_dev_dprtmt_name"), F.col("mv.model_ver_sid"),
    )
)

# Статусы — последняя строка t_model_ver_anlt_dtl по start_dt.
card1_joined = (
    card0.alias("c")
    .join(anlt_raw.alias("a"), F.col("c.model_ver_sid") == F.col("a.model_ver_sid"), "left")
    .select("c.*", F.col("a.model_ver_stts_name"), F.col("a.model_stts_name"), F.col("a.model_ver_prom_expl_flag"), F.col("a.start_dt").alias("__anlt_start_dt"))
)
card1 = take_latest(card1_joined, ["model_ver_sid"], "__anlt_start_dt", "t_model_ver_anlt_dtl").drop("__anlt_start_dt")

# Бизнес-задача — последняя связь по start_dt.
card2_joined = (
    card1.alias("c")
    .join(busn_link_raw.alias("x"), F.col("c.model_ver_sid") == F.col("x.model_ver_sid"), "left")
    .select("c.*", F.col("x.busn_task_sid"), F.col("x.start_dt").alias("__busn_link_start_dt"))
)
card2 = take_latest(card2_joined, ["model_ver_sid"], "__busn_link_start_dt", "t_model_ver_x_busn_task").drop("__busn_link_start_dt")

# Промышленная версия — последняя по start_dt, без фильтра статуса (как в карточке исходника).
card3_joined = (
    card2.alias("c")
    .join(prom_raw.alias("p"), F.col("c.model_ver_sid") == F.col("p.model_ver_sid"), "left")
    .select(
        "c.*", F.col("p.model_ver_prom_instr_name"), F.col("p.model_ver_prom_sys_name"),
        F.col("p.model_ver_prom_sid"), F.col("p.model_ver_prom_stts_name"),
        F.col("p.model_ver_prom_crtn_dttm"), F.col("p.model_ver_prom_dev_block_name"),
        F.col("p.model_ver_prom_dev_dprtmt_name"), F.col("p.start_dt").alias("__prom_start_dt"),
    )
)
card3 = take_latest(card3_joined, ["model_ver_sid"], "__prom_start_dt", "t_model_ver_prom").drop("__prom_start_dt")

# Идентификаторы ручного и автоматического мониторинга — последние связи по start_dt.
card4_joined = (
    card3.alias("c")
    .join(manual_link_raw.alias("x"), F.col("c.model_ver_sid") == F.col("x.model_ver_sid"), "left")
    .select("c.*", F.col("x.montrg_manual_sid"), F.col("x.start_dt").alias("__manual_link_start_dt"))
)
card4 = take_latest(card4_joined, ["model_ver_sid"], "__manual_link_start_dt", "t_model_ver_x_montrg_manual").drop("__manual_link_start_dt")

card5_joined = (
    card4.alias("c")
    .join(auto_link_raw.alias("x"), F.col("c.model_ver_sid") == F.col("x.model_ver_sid"), "left")
    .select("c.*", F.col("x.montrg_auto_sid"), F.col("x.start_dt").alias("__auto_link_start_dt"))
)
card5 = take_latest(card5_joined, ["model_ver_sid"], "__auto_link_start_dt", "t_model_ver_x_montrg_auto").drop("__auto_link_start_dt")

# Валидация — последняя строка по start_dt.
card6_joined = (
    card5.alias("c")
    .join(valid_raw.alias("v"), F.col("c.model_ver_sid") == F.col("v.model_ver_sid"), "left")
    .select(
        "c.*", F.col("v.valid_crtn_dttm"), F.col("v.valid_sid"), F.col("v.valid_report_sid"),
        F.col("v.valid_dprtmt_name"), F.col("v.start_dt").alias("__valid_start_dt"),
    )
)
card6 = take_latest(card6_joined, ["model_ver_sid"], "__valid_start_dt", "t_valid").drop("__valid_start_dt")


In [ ]:
# Атрибуты бизнес-задачи — последняя SCD-строка по start_dt.
card7_joined = (
    card6.alias("c")
    .join(busn_raw.alias("b"), F.col("c.busn_task_sid") == F.col("b.busn_task_sid"), "left")
    .select(
        "c.*", F.col("b.busn_task_name"), F.col("b.busn_task_claim_descr_txt"),
        F.col("b.busn_task_crtn_dttm"), F.col("b.busn_task_employer_block_name"),
        F.col("b.busn_task_employer_dprtmt_name"), F.col("b.start_dt").alias("__busn_start_dt"),
    )
)
card7 = take_latest(card7_joined, ["model_ver_sid"], "__busn_start_dt", "t_busn_task").drop("__busn_start_dt")

# Результат ручного мониторинга — последняя строка по start_dt для уже выбранного monitoring_sid.
card8_joined = (
    card7.alias("c")
    .join(manual_result_raw.alias("r"), F.col("c.montrg_manual_sid") == F.col("r.montrg_manual_sid"), "left")
    .select(
        "c.*", F.col("r.montrg_manual_rslt_end_dttm"), F.col("r.montrg_manual_rslt_start_dttm"),
        F.col("r.montrg_manual_rslt_report_sid"), F.col("r.start_dt").alias("__manual_result_start_dt"),
    )
)
card8 = take_latest(card8_joined, ["model_ver_sid"], "__manual_result_start_dt", "t_montrg_manual_rslt").drop("__manual_result_start_dt")

# Автомониторинг — последняя SCD-строка по start_dt.
card9_joined = (
    card8.alias("c")
    .join(auto_raw.alias("a"), F.col("c.montrg_auto_sid") == F.col("a.montrg_auto_sid"), "left")
    .select(
        "c.*", F.col("a.montrg_auto_crtn_dttm"), F.col("a.montrg_auto_end_dttm"),
        F.col("a.montrg_auto_dprtmt_name"), F.col("a.start_dt").alias("__auto_start_dt"),
    )
)
card9 = take_latest(card9_joined, ["model_ver_sid"], "__auto_start_dt", "t_montrg_auto").drop("__auto_start_dt")

# IT-валидация — последняя строка по start_dt для уже выбранной промышленной версии.
card10_joined = (
    card9.alias("c")
    .join(valid_it_raw.alias("v"), F.col("c.model_ver_prom_sid") == F.col("v.model_ver_prom_sid"), "left")
    .select(
        "c.*", F.col("v.valid_it_end_dttm"), F.col("v.valid_it_start_dttm"),
        F.col("v.valid_it_crtn_dttm"), F.col("v.valid_it_rslt_name"),
        F.col("v.valid_it_dprtmt_name"), F.col("v.start_dt").alias("__valid_it_start_dt"),
    )
)
card10 = take_latest(card10_joined, ["model_ver_sid"], "__valid_it_start_dt", "t_valid_it").drop("__valid_it_start_dt")

CARD_COLUMNS = [
    ("model_name", "MODEL_NAME|Наименование модели"),
    ("model_rsk_flag", "MODEL_RSK_FLAG|Флаг риск-модели"),
    ("model_rsk_type_name", "MODEL_RSK_TYPE_NAME|Тип риска модели"),
    ("model_rsk_sgmnt_name", "MODEL_RSK_SGMNT_NAME|Наименование риск-сегмента модели"),
    ("model_type_name", "MODEL_TYPE_NAME|Тип модели"),
    ("model_subtype_name", "MODEL_SUBTYPE_NAME|Подтип модели"),
    ("model_code", "MODEL_CODE|Код модели"),
    ("model_conf_ctgry_name", "MODEL_CONF_CTGRY_NAME|Категория конфиденциальности информации о модели"),
    ("model_sel_secret_flag", "MODEL_SEL_SECRET_FLAG|Флаг коммерческой тайны"),
    ("model_sid", "MODEL_SID|Идентификатор модели"),
    ("model_ver_signfcnt_lvl_name", "MODEL_VER_SIGNFCNT_LVL_NAME|Наименование степени значимости версии модели"),
    ("model_ver_dev_start_fact_dttm", "MODEL_VER_DEV_START_FACT_DTTM|Фактическая дата-время начала разработки версии модели"),
    ("model_ver_signfcnt_ctgry_code", "MODEL_VER_SIGNFCNT_CTGRY_CODE|Наименование категории значимости версии модели"),
    ("model_ver_signfcnt_ctgry_descr_txt", "MODEL_VER_SIGNFCNT_CTGRY_DESCR_TXT|Обоснование категории значимости версии модели"),
    ("model_ver_dev_end_fact_dttm", "MODEL_VER_DEV_END_FACT_DTTM|Фактическая дата-время окончания разработки версии модели"),
    ("model_ver_dev_sys_name", "MODEL_VER_DEV_SYS_NAME|Система в которой разрабатывалась версия модели"),
    ("model_ver_data_mart_link_txt", "MODEL_VER_DATA_MART_LINK_TXT|Ссылка на витрину данных для обучения версии модели"),
    ("model_ver_crtn_dttm", "MODEL_VER_CRTN_DTTM|Дата-время создания версии модели"),
    ("model_ver_dev_report_sid", "MODEL_VER_DEV_REPORT_SID|Идентификатор отчета о разработке версия модели"),
    ("model_ver_prevalid_report_file_sid", "MODEL_VER_PREVALID_REPORT_FILE_SID|Идентификатор файла с отчетом о превалидации версии модели"),
    ("model_ver_prevalid_report_link_sid", "MODEL_VER_PREVALID_REPORT_LINK_SID|Идентификатор ссылки на отчет о превалидации версии модели"),
    ("model_ver_prevalid_report_link_txt", "MODEL_VER_PREVALID_REPORT_LINK_TXT|Ссылка на отчет о превалидации"),
    ("model_ver_dev_block_name", "MODEL_VER_DEV_BLOCK_NAME|Наименование блока разработки версии модели"),
    ("model_ver_dev_dprtmt_name", "MODEL_VER_DEV_DPRTMT_NAME|Наименование подразделения разработки версии модели"),
    ("model_ver_sid", "MODEL_VER_SID|Идентификатор версии модели"),
    ("model_ver_stts_name", "MODEL_VER_STTS_NAME|Наименование статуса версии модели"),
    ("model_stts_name", "MODEL_STTS_NAME|Статус модели"),
    ("model_ver_prom_expl_flag", "MODEL_VER_PROM_EXPL_FLAG|Флаг нахождения версии модели в эксплуатации"),
    ("busn_task_sid", "BUSN_TASK_SID|Идентификатор бизнес-задачи"),
    ("model_ver_prom_instr_name", "MODEL_VER_PROM_INSTR_NAME|Среда (инструмент) исполнения промышленной версии модели"),
    ("model_ver_prom_sys_name", "MODEL_VER_PROM_SYS_NAME|Система в которой реализована промышленная версия модели"),
    ("model_ver_prom_sid", "MODEL_VER_PROM_SID|Идентификатор промышленной версии модели"),
    ("model_ver_prom_stts_name", "MODEL_VER_PROM_STTS_NAME|Статус промышленной версии модели"),
    ("model_ver_prom_crtn_dttm", "MODEL_VER_PROM_CRTN_DTTM|Дата-время создания промышленной версии модели"),
    ("model_ver_prom_dev_block_name", "MODEL_VER_PROM_DEV_BLOCK_NAME|Блок разработки промышленной версии модели"),
    ("model_ver_prom_dev_dprtmt_name", "MODEL_VER_PROM_DEV_DPRTMT_NAME|Подразделение разработки промышленной версии модели"),
    ("montrg_manual_sid", "MONTRG_MANUAL_SID|Идентификатор ручного мониторинга"),
    ("montrg_auto_sid", "MONTRG_AUTO_SID|Идентификатор автоматического мониторинга"),
    ("valid_crtn_dttm", "VALID_CRTN_DTTM|Дата-время создания валидации"),
    ("valid_sid", "VALID_SID|Идентификатор валидации"),
    ("valid_report_sid", "VALID_REPORT_SID|Идентификатор отчета о валидации"),
    ("valid_dprtmt_name", "VALID_DPRTMT_NAME|Подразделение, проводящее валидацию"),
    ("busn_task_name", "BUSN_TASK_NAME|Наименование бизнес-задачи"),
    ("busn_task_claim_descr_txt", "BUSN_TASK_CLAIM_DESCR_TXT|Описание требований бизнес-задачи"),
    ("busn_task_crtn_dttm", "BUSN_TASK_CRTN_DTTM|Дата-время создания бизнес-задачи"),
    ("busn_task_employer_block_name", "BUSN_TASK_EMPLOYER_BLOCK_NAME|Блок заказчика бизнес-задачи"),
    ("busn_task_employer_dprtmt_name", "BUSN_TASK_EMPLOYER_DPRTMT_NAME|Наименование подразделения заказчика бизнес-задачи"),
    ("montrg_manual_rslt_end_dttm", "MONTRG_MANUAL_RSLT_END_DTTM|Дата-время окончания процесса ручного мониторинга в рамках которого получен текущий результат"),
    ("montrg_manual_rslt_start_dttm", "MONTRG_MANUAL_RSLT_START_DTTM|Дата-время начала процесса ручного мониторинга в рамках которого получен текущий результат"),
    ("montrg_manual_rslt_report_sid", "MONTRG_MANUAL_RSLT_REPORT_SID|Идентификатор отчёта о результате ручного мониторинга"),
    ("montrg_auto_crtn_dttm", "MONTRG_AUTO_CRTN_DTTM|Дата-время создания автоматического мониторинга"),
    ("montrg_auto_end_dttm", "MONTRG_AUTO_END_DTTM|Дата-время окончания мониторинга"),
    ("montrg_auto_dprtmt_name", "MONTRG_AUTO_DPRTMT_NAME|Наименование подразделения проводящего автоматического мониторинга"),
    ("valid_it_end_dttm", "VALID_IT_END_DTTM|Дата-время окончания ИТ-валидации"),
    ("valid_it_start_dttm", "VALID_IT_START_DTTM|Дата-время начала ИТ-валидации"),
    ("valid_it_crtn_dttm", "VALID_IT_CRTN_DTTM|Дата-время создания ИТ-валидации"),
    ("valid_it_rslt_name", "VALID_IT_RSLT_NAME|Результат ИТ-валидации"),
    ("valid_it_dprtmt_name", "VALID_IT_DPRTMT_NAME|Подразделение проводящее ИТ-валидацию"),
]

card_full = card10.select(*[F.col(source).alias(target) for source, target in CARD_COLUMNS])


## Периметр: все версии моделей в эксплуатации


In [ ]:
CARD_MODEL_VER_COLUMN = "MODEL_VER_SID|Идентификатор версии модели"
EXPLOITATION_FLAG_COLUMN = "MODEL_VER_PROM_EXPL_FLAG|Флаг нахождения версии модели в эксплуатации"

card_full_count = card_full.count()
card_duplicate_count = (
    card_full.groupBy(F.col(f"`{CARD_MODEL_VER_COLUMN}`"))
    .count()
    .filter(F.col("count") > 1)
    .count()
)
if card_duplicate_count:
    raise RuntimeError(f"Карточка до отбора содержит дубликаты model_ver_sid: {card_duplicate_count}")


def as_true(column_name: str):
    return F.col(f"`{column_name}`").cast("boolean") == F.lit(True)


# Единственный критерий отбора: актуальный признак эксплуатации.
# Категория значимости не фильтруется и не участвует в формировании периметра.
operational_card_sdf = card_full.filter(as_true(EXPLOITATION_FLAG_COLUMN)).cache()
operational_count = operational_card_sdf.count()
if operational_count == 0:
    raise RuntimeError("В Библиотеке моделей не найдено версий с model_ver_prom_expl_flag = true")

output_pdf = operational_card_sdf.toPandas()
if output_pdf[CARD_MODEL_VER_COLUMN].duplicated().any():
    raise RuntimeError("В периметре эксплуатации появились дубликаты model_ver_sid")

output_pdf.to_excel(OUTPUT_DIR / "output.xlsx", index=False)
missingness_pdf = pd.DataFrame(
    [(name, int(output_pdf[name].isna().sum())) for name in output_pdf.columns],
    columns=["Key", "Value"],
)
missingness_pdf.to_excel(OUTPUT_DIR / "example.xlsx", index=False)

# Все последующие выгрузки получают тот же периметр напрямую из Spark.
model_sample_sdf = (
    operational_card_sdf
    .select(F.col(f"`{CARD_MODEL_VER_COLUMN}`").cast("string").alias("model_ver_sid"))
    .dropDuplicates()
)
selected_model_ids_pdf = model_sample_sdf.toPandas()

print("Карточка до отбора:", card_full_count)
print("Версий в эксплуатации по model_ver_prom_expl_flag:", operational_count)
print("Итог output.xlsx:", len(output_pdf))


## Истории статусов и параметры — по окончательно выбранным моделям и входным ID


In [ ]:
def cap_exact_open_end(column_name: str):
    return F.when(
        F.col(column_name).cast("string") == "9999-12-31 00:00:00",
        F.current_timestamp(),
    ).otherwise(F.col(column_name).cast("timestamp"))


BUSINESS_STATUS_LABELS = {
    "BUSINESS_TASK_BACKLOG": "Ожидает начала (бэклог)",
    "BUSINESS_TASK_DEVELOPMENT": "В работе",
    "BUSINESS_TASK_DECISION_MAKING": "Принятие решения о завершении задачи",
    "BUSINESS_TASK_DONE": "Завершена",
}
IMPLEMENTATION_STATUS_LABELS = {
    "IMPLEMENTATION_PREPARATION_PILOT": "Подготовка к пилоту",
    "IMPLEMENTATION_PREPARATION_EXPLOITATION": "Подготовка в эксплуатации",
    "IMPLEMENTATION_EXPLOITATION": "Эксплуатация",
    "IMPLEMENTATION_PILOT_DONE": "Пилот завершен",
    "IMPLEMENTATION_WAITING_VALIDATION": "Ожидает валидации",
    "IMPLEMENTATION_EXPLOITATION_WITHOUT_VALIDATION": "Эксплуатация (без валдиации)",
    "IMPLEMENTATION_WAITING_IT_VALIDATION": "Ожидает IT-валидации",
    "IMPLEMENTATION_FORMATION": "Формирование",
    "IMPLEMENTATION_EXPLOITATION_WITHDRAWN": "Выведено из эксплуатации",
    "IMPLEMENTATION_PILOT": "Пилот",
    "IMPLEMENTATION_CANCELED": "Отмененно",
}


# Связанные бизнес-задачи и внедрения формируются напрямую из Hive
# только для окончательно выбранных model_ver_sid. Внешние списки ID не используются.
selected_busn_history_joined = (
    model_sample_sdf.alias("s")
    .join(busn_link_raw.alias("x"), F.col("s.model_ver_sid") == F.col("x.model_ver_sid").cast("string"), "left")
    .select(F.col("s.model_ver_sid"), F.col("x.busn_task_sid").cast("string").alias("busn_task_sid"), F.col("x.start_dt").alias("__link_start_dt"))
)
selected_busn_history_ids = (
    take_latest(selected_busn_history_joined, ["model_ver_sid"], "__link_start_dt", "история: бизнес-задача")
    .filter(F.col("busn_task_sid").isNotNull())
    .select("busn_task_sid")
    .distinct()
)
business_task_input_sdf = selected_busn_history_ids.select(
    F.col("busn_task_sid").cast("string").alias("busn_task_sid")
)

selected_prom_history_joined = (
    model_sample_sdf.alias("s")
    .join(
        prom_raw.alias("p"),
        (F.col("s.model_ver_sid") == F.col("p.model_ver_sid").cast("string"))
        & (F.col("p.model_ver_prom_stts_name") == "MODEL_PROM_VERSION_EXPLOIT_PERMITTED"),
        "left",
    )
    .select(F.col("s.model_ver_sid"), F.col("p.model_ver_prom_sid"), F.col("p.start_dt").alias("__prom_start_dt"))
)
selected_prom_history = take_latest(
    selected_prom_history_joined, ["model_ver_sid"], "__prom_start_dt", "история: промверсия"
).drop("__prom_start_dt")
selected_implm_history_joined = (
    selected_prom_history.alias("p")
    .join(prom_link_raw.alias("x"), F.col("p.model_ver_prom_sid") == F.col("x.model_ver_prom_sid"), "left")
    .select(F.col("p.model_ver_sid"), F.col("x.prom_implm_sid").cast("string").alias("implm_sid"), F.col("x.start_dt").alias("__implm_start_dt"))
)
selected_implm_history_ids = (
    take_latest(selected_implm_history_joined, ["model_ver_sid"], "__implm_start_dt", "история: внедрение")
    .filter(F.col("implm_sid").isNotNull())
    .select("implm_sid")
    .distinct()
)
implementation_input_sdf = selected_implm_history_ids.select(
    F.col("implm_sid").cast("string").alias("implm_sid")
)

print("Связанных business_task_sid:", business_task_input_sdf.count())
print("Связанных implm_sid:", implementation_input_sdf.count())


def mapped_status(column_name: str, mapping):
    expression = F.col(column_name)
    for source, target in mapping.items():
        expression = F.when(F.col(column_name) == source, F.lit(target)).otherwise(expression)
    return expression


business_change = (
    change_raw.filter(
        (F.col("ent_type_name") == "BUSINESS_TASK")
        & (F.col("start_dttm") < F.to_timestamp(F.lit(HISTORY_CUTOFF)))
        & (F.col("ent_param_chg_sid") == "BUSINESS_TASK_STATUS")
    )
)
busn_task_life_stage = (
    business_task_input_sdf.alias("ids")
    .join(business_change.alias("c"), F.col("ids.busn_task_sid") == F.col("c.ent_sid").cast("string"), "left")
    .select(
        "ids.*",
        mapped_status("c.ent_param_chg_val", BUSINESS_STATUS_LABELS).alias("life_stage_name"),
        F.col("c.start_dttm").alias("start_dttm"),
        cap_exact_open_end("c.end_dttm").alias("end_dttm"),
    )
)

implementation_change = change_raw.filter(
    (F.col("start_dttm") < F.to_timestamp(F.lit(HISTORY_CUTOFF)))
    & (F.col("ent_param_chg_sid") == "IMPLEMENTATION_STATUS")
)
implm_life_stage = (
    implementation_input_sdf.alias("ids")
    .join(implementation_change.alias("c"), F.col("ids.implm_sid") == F.col("c.ent_sid").cast("string"), "left")
    .select(
        "ids.*",
        mapped_status("c.ent_param_chg_val", IMPLEMENTATION_STATUS_LABELS).alias("life_stage_name"),
        F.col("c.start_dttm").alias("start_dttm"),
        cap_exact_open_end("c.end_dttm").alias("end_dttm"),
    )
)

# Два разных параметра сохранены под исходными названиями old/new — это часть структуры исходного результата.
old_param_joined = (
    model_sample_sdf.alias("s")
    .join(
        change_raw.filter(F.col("ent_param_chg_sid") == "MODEL_VERSION_IMPORTANCE_UMR").alias("c"),
        F.col("s.model_ver_sid") == F.col("c.ent_sid").cast("string"),
        "inner",
    )
    .select(F.col("s.model_ver_sid"), F.col("c.ent_param_chg_val").alias("old_import_umr"), F.col("c.start_dttm").alias("__param_start_dttm"))
)
old_param = take_latest(old_param_joined, ["model_ver_sid"], "__param_start_dttm", "MODEL_VERSION_IMPORTANCE_UMR").drop("__param_start_dttm")

new_param_joined = (
    model_sample_sdf.alias("s")
    .join(
        change_raw.filter(F.col("ent_param_chg_sid") == "MODEL_VERSION_MODEL_EFFECT").alias("c"),
        F.col("s.model_ver_sid") == F.col("c.ent_sid").cast("string"),
        "inner",
    )
    .select(
        F.col("s.model_ver_sid"), F.col("c.ent_param_chg_val").alias("new_import_umr"),
        F.col("c.ent_param_chg_usr_name").alias("chg_usr_name"),
        F.col("c.start_dttm").alias("data"), F.col("c.start_dttm").alias("__param_start_dttm"),
    )
)
new_param = take_latest(new_param_joined, ["model_ver_sid"], "__param_start_dttm", "MODEL_VERSION_MODEL_EFFECT").drop("__param_start_dttm")
model_with_old_new_importance = new_param.join(old_param, "model_ver_sid", "left").select(
    "model_ver_sid", "old_import_umr", "new_import_umr", "chg_usr_name", "data"
)


## Временная шкала — те же MIN/MAX по датам изменений


In [ ]:
def change_stage(base: DataFrame, entity_column: str, parameter_sids, stage_name: str) -> DataFrame:
    params = [parameter_sids] if isinstance(parameter_sids, str) else list(parameter_sids)
    return (
        base.alias("b")
        .join(
            change_raw.filter(F.col("ent_param_chg_sid").isin(*params)).alias("c"),
            F.col(f"b.{entity_column}") == F.col("c.ent_sid"),
            "inner",
        )
        .groupBy(F.col("b.model_ver_sid").alias("model_ver_sid"))
        .agg(F.min("c.start_dttm").alias("start_dt"), F.max("c.start_dttm").alias("end_dt"))
        .withColumn("life_cycle_stage", F.when(F.col("start_dt").isNotNull() & F.col("end_dt").isNotNull(), F.lit(stage_name)))
        .select("model_ver_sid", "start_dt", "end_dt", "life_cycle_stage")
    )


sample_ids = model_sample_sdf.select(F.col("model_ver_sid").cast("string").alias("model_ver_sid"))

busn_link_joined = (
    sample_ids.alias("s")
    .join(busn_link_raw.alias("x"), F.col("s.model_ver_sid") == F.col("x.model_ver_sid").cast("string"), "left")
    .select(F.col("s.model_ver_sid"), F.col("x.busn_task_sid"), F.col("x.start_dt").alias("__link_start_dt"))
)
busn_for_stage = take_latest(busn_link_joined, ["model_ver_sid"], "__link_start_dt", "жизненный цикл: бизнес-задача").drop("__link_start_dt")
stage_business = change_stage(busn_for_stage, "busn_task_sid", "BUSINESS_TASK_STATUS", "Постановка бизнес-задачи")

sample_version_entity = sample_ids.select("model_ver_sid", F.col("model_ver_sid").alias("entity_sid"))
stage_development = change_stage(sample_version_entity, "entity_sid", "MODEL_VERSION_STATUS", "Разработка модели")

stage_prevalid = (
    sample_ids.alias("s")
    .join(model_ver_raw.alias("v"), F.col("s.model_ver_sid") == F.col("v.model_ver_sid").cast("string"), "left")
    .groupBy(F.col("s.model_ver_sid").alias("model_ver_sid"))
    .agg(F.min("v.model_ver_prevalid_dttm").alias("start_dt"), F.max("v.model_ver_prevalid_dttm").alias("end_dt"))
    .withColumn("life_cycle_stage", F.when(F.col("start_dt").isNotNull() & F.col("end_dt").isNotNull(), F.lit("Превалидация модели")))
    .select("model_ver_sid", "start_dt", "end_dt", "life_cycle_stage")
)

valid_joined = (
    sample_ids.alias("s")
    .join(valid_raw.alias("v"), F.col("s.model_ver_sid") == F.col("v.model_ver_sid").cast("string"), "left")
    .select(F.col("s.model_ver_sid"), F.col("v.valid_sid"), F.col("v.start_dt").alias("__valid_start_dt"))
)
valid_for_stage = take_latest(valid_joined, ["model_ver_sid"], "__valid_start_dt", "жизненный цикл: валидация").drop("__valid_start_dt")
stage_validation = change_stage(valid_for_stage, "valid_sid", "VALIDATION_STATUS", "Валидация модели")

auto_link_joined = (
    sample_ids.alias("s")
    .join(auto_link_raw.alias("x"), F.col("s.model_ver_sid") == F.col("x.model_ver_sid").cast("string"), "left")
    .select(F.col("s.model_ver_sid"), F.col("x.montrg_auto_sid"), F.col("x.start_dt").alias("__auto_link_start_dt"))
)
auto_for_stage = take_latest(auto_link_joined, ["model_ver_sid"], "__auto_link_start_dt", "жизненный цикл: автомониторинг").drop("__auto_link_start_dt")
stage_auto = change_stage(auto_for_stage, "montrg_auto_sid", "MONITORING_STATUS", "Модель поставлена на автомониторинг")

# Для этапов промверсии/IT-валидации/внедрения исходник использует только разрешённую к эксплуатации промверсию.
prom_permitted_joined = (
    sample_ids.alias("s")
    .join(
        prom_raw.alias("p"),
        (F.col("s.model_ver_sid") == F.col("p.model_ver_sid").cast("string"))
        & (F.col("p.model_ver_prom_stts_name") == "MODEL_PROM_VERSION_EXPLOIT_PERMITTED"),
        "left",
    )
    .select(F.col("s.model_ver_sid"), F.col("p.model_ver_prom_sid"), F.col("p.start_dt").alias("__prom_start_dt"))
)
prom_for_stage = take_latest(prom_permitted_joined, ["model_ver_sid"], "__prom_start_dt", "жизненный цикл: промверсия").drop("__prom_start_dt")
stage_prom = change_stage(prom_for_stage, "model_ver_prom_sid", "MODEL_PROM_VERSION_STATUS", "Разработка промышленной версии модели")

valid_it_joined = (
    prom_for_stage.alias("p")
    .join(valid_it_raw.alias("v"), F.col("p.model_ver_prom_sid") == F.col("v.model_ver_prom_sid"), "left")
    .select(F.col("p.model_ver_sid"), F.col("v.valid_it_sid"), F.col("v.start_dt").alias("__valid_it_start_dt"))
)
valid_it_for_stage = take_latest(valid_it_joined, ["model_ver_sid"], "__valid_it_start_dt", "жизненный цикл: IT-валидация").drop("__valid_it_start_dt")
stage_it_validation = change_stage(valid_it_for_stage, "valid_it_sid", "IT_VALIDATION_STATUS", "Начало IT валидация")

implm_link_joined = (
    prom_for_stage.alias("p")
    .join(prom_link_raw.alias("x"), F.col("p.model_ver_prom_sid") == F.col("x.model_ver_prom_sid"), "left")
    .select(F.col("p.model_ver_sid"), F.col("x.prom_implm_sid"), F.col("x.start_dt").alias("__implm_link_start_dt"))
)
implm_for_stage = take_latest(implm_link_joined, ["model_ver_sid"], "__implm_link_start_dt", "жизненный цикл: внедрение").drop("__implm_link_start_dt")
stage_implementation = change_stage(
    implm_for_stage,
    "prom_implm_sid",
    ["IMPLEMENTATION_STATUS", "IMPLEMENTATION_PILOT_CRITERIA"],
    "Внедрение модели",
)

model_lifecycle_all = reduce(
    lambda left, right: left.unionByName(right),
    [stage_business, stage_development, stage_prevalid, stage_validation, stage_auto, stage_prom, stage_it_validation, stage_implementation],
)
model_lifecycle = model_lifecycle_all.filter(F.col("life_cycle_stage").isNotNull())
lifecycle_bounds = model_lifecycle.groupBy("model_ver_sid").agg(
    F.min("start_dt").alias("min(start_dt)"),
    F.max("end_dt").alias("max(end_dt)"),
)


## Контроль совпадения периметра и выгрузка


In [ ]:
def to_pandas_limited(df: DataFrame, name: str) -> pd.DataFrame:
    result = df.limit(MAX_EXCEL_ROWS + 1).toPandas()
    if len(result) > MAX_EXCEL_ROWS:
        raise RuntimeError(f"{name}: превышен лимит строк Excel")
    return result


def save_spark_excel(df: DataFrame, filename: str) -> None:
    to_pandas_limited(df, filename).to_excel(OUTPUT_DIR / filename, index=False)
    print("Сохранён:", OUTPUT_DIR / filename)


save_spark_excel(busn_task_life_stage, "busn_task_life_stage.xlsx")
save_spark_excel(implm_life_stage, "df_implm_life_stage.xlsx")
save_spark_excel(model_with_old_new_importance, "model_with_old_new_importance_umr.xlsx")
save_spark_excel(model_lifecycle, "data_to_process_mining.xlsx")
save_spark_excel(lifecycle_bounds, "data_life_stage.xlsx")

summary_pdf = pd.DataFrame(
    [
        ("карточка до отбора", card_full_count),
        ("версии в эксплуатации по model_ver_prom_expl_flag", operational_count),
        ("output.xlsx: все версии в эксплуатации", len(output_pdf)),
        ("строки для расчёта заполненности", len(output_pdf)),
    ],
    columns=["check", "value"],
)

with pd.ExcelWriter(OUTPUT_DIR / "selection_control.xlsx", engine="openpyxl") as writer:
    summary_pdf.to_excel(writer, sheet_name="summary", index=False)
    selected_model_ids_pdf.to_excel(writer, sheet_name="selected_model_ids", index=False)

print("Сохранены output.xlsx, example.xlsx и selection_control.xlsx")
print("Контроль периметра пройден: включены все версии с model_ver_prom_expl_flag = true")

# Освобождаем cache после завершения всех выгрузок.
operational_card_sdf.unpersist()


## Правило формирования периметра

- внешние файлы со списком версий и статусами внедрения полностью исключены из формирования выборки;
- эксплуатация определяется непосредственно по актуальному полю Библиотеки моделей `model_ver_prom_expl_flag = true`;
- отбор по категории значимости отсутствует: A, B, C, D, E, пустые и прочие категории сохраняются, если версия находится в эксплуатации;
- связанные бизнес-задачи и внедрения выводятся из Hive по всем выбранным `model_ver_sid`; внешние Excel-файлы для них не нужны;
- остальная логика карточки, SCD-отбора по `start_dt`, историй и выгрузок сохранена.


In [ ]:
# Закрывайте Spark только если после ноутбука сессия больше не нужна.
# spark.stop()
